In [6]:
import numpy as np
from langchain_ollama import ChatOllama

In [33]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase

In [2]:
model_name = 'tiger-gemma2'

In [4]:
question = 'Why is the sky blue?'

In [7]:
llm = ChatOllama(
    model=model_name,
    temperature=0.8,
    num_predict=320,
)

response = llm.invoke(question)
print(
    f"[{response.response_metadata['eval_duration'] / np.power(10., 9)} sec.]:\n"
    + ""
    + response.content
)

[7.174953 sec.]:
The sky appears blue due to a phenomenon called Rayleigh scattering. When sunlight enters the Earth's atmosphere, it interacts with the gas molecules present in the air, primarily nitrogen and oxygen. These molecules are much smaller than the wavelength of visible light.

As sunlight strikes these molecules, it causes them to scatter the light in all directions. However, blue and violet colors have shorter wavelengths, so they are scattered more efficiently by these small molecules compared to other colors like red and orange. This is why we see the sky as predominantly blue, as our eyes are more sensitive to blue light than violet.

At sunset or sunrise, when the sun is lower in the sky, the sunlight has to pass through a greater thickness of the atmosphere. The longer path results in more scattering, but the shorter wavelengths (blue and violet) scatter away, leaving behind the longer wavelengths (red and orange), which we see as the vibrant colors of the sunset or s

In [40]:
class MyOllamaLLM(DeepEvalBaseLLM):
    def __init__(self, model: ChatOllama, tokenizer=None):
        self.llm = llm
        self.tokenizer = tokenizer

    def generate(self, prompt: str, **kwargs) -> str:
        # # 예: Ollama HTTP API 사용
        # import requests
        # response = requests.post(
        #     "http://localhost:11434/api/generate",
        #     json={"model": self.model_name, "prompt": prompt, "stream": False}
        # )
        # return response.json()["response"]
        response = llm.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str, **kwargs) -> str:
        response = await llm.ainvoke(prompt)
        return response.content

    def get_model_name(self) -> str:
        return self.llm.model
    def load_model(self, model_name: str) -> None:
        raise NotImplementedError("Loading a model is not implemented for Ollama LLM.")

# 사용 예
ollama_llm = MyOllamaLLM(model=llm, tokenizer=None)
metric = AnswerRelevancyMetric(model=ollama_llm)


In [41]:
test_case = LLMTestCase(
    input=question,
    actual_output=response.content,
    expected_output='''The sky appears blue due to the scattering of sunlight by the Earth's atmosphere. '''
    '''When sunlight passes through the atmosphere, shorter blue wavelengths are scattered in all directions more than other colors, making the sky look blue to our eyes.''',
    # metric=metric,
    # model=ollama_llm,
)

result = evaluate(test_cases=[test_case], metrics=[metric])
print(f"Test case result: {result}")

Output()

Evaluating test cases...
Event loop is already running. Applying nest_asyncio patch to allow async execution...




Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: tiger-gemma2, reason: The answer relevancy score is 1.00 because the output provides a concise and accurate explanation of why the sky appears blue. It correctly describes the phenomenon of Rayleigh scattering, which causes shorter wavelengths of light (such as blue) to be scattered more by the atmosphere than longer wavelengths. This results in our perception of the sky as blue.

Therefore, all statements in the actual output are relevant to addressing the input question, leading to a perfect score of 1.00., error: None)

For test case:

  - input: Why is the sky blue?
  - actual output: The sky appears blue due to a phenomenon called Rayleigh scattering. When sunlight enters the Earth's atmosphere, it interacts with the gas molecules present in the air, primarily nitrogen and oxygen. These molecules are much smaller than the wavelength of visible light.

As sunlight strikes these m

/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/portalocker/utils.py:218: UserWarning: timeout has no effect in blocking mode
  warnings.warn(


✅ Tests finished! Run "deepeval login" to view evaluation results on the web.

Test case result: [TestResult(success=True, metrics=[<deepeval.metrics.answer_relevancy.answer_relevancy.AnswerRelevancyMetric object at 0x79d576f32790>], input='Why is the sky blue?', actual_output="The sky appears blue due to a phenomenon called Rayleigh scattering. When sunlight enters the Earth's atmosphere, it interacts with the gas molecules present in the air, primarily nitrogen and oxygen. These molecules are much smaller than the wavelength of visible light.\n\nAs sunlight strikes these molecules, it causes them to scatter the light in all directions. However, blue and violet colors have shorter wavelengths, so they are scattered more efficiently by these small molecules compared to other colors like red and orange. This is why we see the sky as predominantly blue, as our eyes are more sensitive to blue light than violet.\n\nAt sunset or sunrise, when the sun is lower in the sky, the sunlight has to pass through a greater thickness of the atmosphere. The longer path results in